---
## PART 1 — SETUP
---

### Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted!')

### Cell 2 — Install Libraries

In [ ]:
!pip install tf-keras-vis -q
print('✅ tf-keras-vis installed!')

### Cell 3 — Import All Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print('TF Version:', tf.__version__)
print('✅ All libraries imported!')

### Cell 4 — Set Paths

In [ ]:
DATASET_PATH   = '/content/drive/MyDrive/OncoAi Dataset'
WEIGHTS_PATH   = os.path.join(DATASET_PATH, 'oncoai_best.weights.h5')
CANCER_PATH    = os.path.join(DATASET_PATH, 'CANCER')
NONCANCER_PATH = os.path.join(DATASET_PATH, 'NON CANCER')
GRADCAM_SAVE   = os.path.join(DATASET_PATH, 'gradcam_samples.png')

IMG_SIZE    = (224, 224)
CLASS_NAMES = ['CANCER', 'NON CANCER']

print('Weights file exists:', os.path.exists(WEIGHTS_PATH))
print('✅ Paths set!')

---
## PART 2 — REBUILD MODEL AND LOAD WEIGHTS
---

### Cell 5 — Rebuild MobileNetV2 Model

In [ ]:
tf.keras.backend.clear_session()

inputs     = tf.keras.Input(shape=(224, 224, 3))
base_model = MobileNetV2(weights=None, include_top=False, input_tensor=inputs)

x       = base_model.output
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
x       = layers.Dense(64, activation='relu')(x)
x       = layers.Dropout(0.2)(x)
outputs = layers.Dense(2, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.load_weights(WEIGHTS_PATH)

print('✅ Model loaded with trained weights!')

# Verify with dummy input
dummy = np.zeros((1, 224, 224, 3))
out   = model.predict(dummy, verbose=0)
print(f'   Test prediction → CANCER:{out[0][0]:.3f} NON CANCER:{out[0][1]:.3f}')

---
## PART 3 — GRAD-CAM IMPLEMENTATION
---

### Cell 6 — Grad-CAM Core Function

In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name, pred_index=None):
    """
    Generates Grad-CAM heatmap for a given image.
    
    Args:
        model: trained Keras model
        img_array: preprocessed image (1, 224, 224, 3)
        last_conv_layer_name: name of last conv layer in base model
        pred_index: class index to visualize (None = predicted class)
    
    Returns:
        heatmap: 2D numpy array (0 to 1)
    """
    # Create a model that outputs last conv layer + final predictions
    grad_model = tf.keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradients of predicted class w.r.t. conv layer output
    grads = tape.gradient(class_channel, conv_outputs)

    # Pool gradients over all axes except channels
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight conv outputs by pooled gradients
    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)

    # Normalize to 0-1
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_path, heatmap, alpha=0.4):
    """
    Overlays Grad-CAM heatmap on original image.
    
    Returns:
        superimposed_img: RGB image with heatmap overlay
    """
    # Load original image
    img = cv2.imread(img_path)
    img = cv2.resize(img, IMG_SIZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize heatmap to image size
    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)

    # Convert to colormap (jet = blue→green→red)
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    # Overlay
    superimposed = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)
    return img, heatmap_resized, superimposed


def preprocess_for_gradcam(img_path):
    """Preprocess image same way as training."""
    img   = load_img(img_path, target_size=IMG_SIZE)
    arr   = img_to_array(img)
    arr   = tf.keras.applications.mobilenet_v2.preprocess_input(arr)
    return np.expand_dims(arr, axis=0)


# Find the last conv layer name in MobileNetV2
last_conv_layer = None
for layer in reversed(base_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer.name
        break

print(f'✅ Grad-CAM functions defined!')
print(f'   Last conv layer: {last_conv_layer}')

---
## PART 4 — GENERATE AND VISUALIZE GRAD-CAM
---

### Cell 7 — Test Grad-CAM on One Image

In [ ]:
import random

# Pick one cancer image to test
cancer_files = [f for f in os.listdir(CANCER_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))]
test_img_name = random.choice(cancer_files)
test_img_path = os.path.join(CANCER_PATH, test_img_name)

print(f'Testing on: {test_img_name}')

# Preprocess and predict
img_array    = preprocess_for_gradcam(test_img_path)
predictions  = model.predict(img_array, verbose=0)
pred_index   = np.argmax(predictions[0])
pred_class   = CLASS_NAMES[pred_index]
confidence   = predictions[0][pred_index] * 100

print(f'Prediction  : {pred_class}')
print(f'Confidence  : {confidence:.1f}%')
print(f'CANCER      : {predictions[0][0]*100:.1f}%')
print(f'NON CANCER  : {predictions[0][1]*100:.1f}%')

# Generate heatmap
heatmap = get_gradcam_heatmap(model, img_array, last_conv_layer, pred_index)
original, heatmap_only, overlay = overlay_gradcam(test_img_path, heatmap)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'Grad-CAM Test — Predicted: {pred_class} ({confidence:.1f}%)',
             fontsize=14, fontweight='bold')

axes[0].imshow(original)
axes[0].set_title('Original Image', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(heatmap_only, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title('Overlay (Red = High Attention)', fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()
print('✅ Grad-CAM test complete!')

### Cell 8 — Generate Grad-CAM for Multiple Cancer Images

In [ ]:
# Show Grad-CAM for 4 cancer + 4 non cancer images
noncancer_files = [f for f in os.listdir(NONCANCER_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))]

cancer_samples    = random.sample(cancer_files, 4)
noncancer_samples = random.sample(noncancer_files, 4)

fig, axes = plt.subplots(8, 3, figsize=(15, 32))
fig.suptitle('ONCOAi — Grad-CAM Heatmaps\n(Red/Yellow = Suspicious Region, Blue = Normal)',
             fontsize=15, fontweight='bold', y=1.01)

all_samples = [
    (CANCER_PATH,    f, 'CANCER',     'red')   for f in cancer_samples
] + [
    (NONCANCER_PATH, f, 'NON CANCER', 'green') for f in noncancer_samples
]

for row, (folder, fname, true_label, color) in enumerate(all_samples):
    img_path  = os.path.join(folder, fname)
    img_arr   = preprocess_for_gradcam(img_path)
    preds     = model.predict(img_arr, verbose=0)
    p_idx     = np.argmax(preds[0])
    p_class   = CLASS_NAMES[p_idx]
    conf      = preds[0][p_idx] * 100

    hmap                   = get_gradcam_heatmap(model, img_arr, last_conv_layer, p_idx)
    original, hmap_only, overlay = overlay_gradcam(img_path, hmap)

    correct = '✅' if p_class == true_label else '❌'

    axes[row][0].imshow(original)
    axes[row][0].set_title(f'True: {true_label}', color=color, fontweight='bold', fontsize=10)
    axes[row][0].axis('off')

    axes[row][1].imshow(hmap_only, cmap='jet')
    axes[row][1].set_title('Heatmap', fontweight='bold', fontsize=10)
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay)
    axes[row][2].set_title(f'{correct} Pred: {p_class} ({conf:.1f}%)',
                            fontweight='bold', fontsize=10)
    axes[row][2].axis('off')

plt.tight_layout()
plt.savefig(GRADCAM_SAVE, dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ Grad-CAM grid saved to: {GRADCAM_SAVE}')

In [ ]:
gradcam_code = '''
import numpy as np
import cv2
import tensorflow as tf
from PIL import Image

IMG_SIZE = (224, 224)
CLASS_NAMES = ["CANCER", "NON CANCER"]

def get_last_conv_layer(base_model):
    for layer in reversed(base_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    return None

def generate_gradcam(model, pil_image, pred_index, last_conv_layer_name):
    """
    Generates Grad-CAM heatmap overlay on image.
    
    Args:
        model: trained Keras model
        pil_image: PIL Image object
        pred_index: predicted class index (0=CANCER, 1=NON CANCER)
        last_conv_layer_name: last conv layer name in base model
    
    Returns:
        overlay_pil: PIL Image with heatmap overlay
        heatmap: raw heatmap numpy array
    """
    # Preprocess
    img  = pil_image.convert("RGB").resize(IMG_SIZE)
    arr  = np.array(img, dtype=np.float32)
    arr  = tf.keras.applications.mobilenet_v2.preprocess_input(arr)
    arr  = np.expand_dims(arr, axis=0)

    # Grad-CAM
    grad_model = tf.keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(arr)
        class_channel = predictions[:, pred_index]

    grads       = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    heatmap      = heatmap.numpy()

    # Overlay
    img_cv       = np.array(img)
    img_cv       = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR)
    hmap_resized = cv2.resize(heatmap, IMG_SIZE)
    hmap_colored = np.uint8(255 * hmap_resized)
    hmap_colored = cv2.applyColorMap(hmap_colored, cv2.COLORMAP_JET)
    superimposed  = cv2.addWeighted(img_cv, 0.6, hmap_colored, 0.4, 0)
    superimposed  = cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)
    overlay_pil   = Image.fromarray(superimposed)

    return overlay_pil, heatmap
'''

# Save to Drive
gradcam_py_path = os.path.join(DATASET_PATH, 'gradcam.py')
with open(gradcam_py_path, 'w') as f:
    f.write(gradcam_code)

print(f'✅ gradcam.py saved to Drive!')
print(f'   Path: {gradcam_py_path}')

### Cell 10 — Download All Files

In [ ]:
from google.colab import files

print('Downloading files...')

print('1/2 Downloading gradcam_samples.png ...')
files.download(GRADCAM_SAVE)

print('2/2 Downloading gradcam.py ...')
files.download(gradcam_py_path)

print('\n✅ Done! Send these to Shabin:')
print('  gradcam.py          → OncoAi/app/gradcam.py')
print('  gradcam_samples.png → OncoAi/reports/figures/gradcam_samples.png')